# Brain MRI and EEG Analysis for Brain Tumor Detection
### CIE-2 Digital Image and Video Processing Academic Project
**Author:** Academic Project Prototype  
**Tech Stack:** OpenCV (`cv2`), NumPy, Pandas, Matplotlib, Scikit-learn  

---
> **Academic / Research Disclaimer:**  
> This notebook demonstrates an academic prototype for digital image processing and neurological signal feature extraction. It is **NOT** a clinical diagnostic tool. MRI image processing serves as the primary tumor-candidate localization method. EEG analysis is supplementary and **cannot** independently confirm or diagnose a brain tumor.

## 1. Import Libraries
We import standard, lightweight libraries for image processing, numerical computation, signal analysis, and machine learning.

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

# Set matplotlib aesthetic styling
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['image.cmap'] = 'gray'
print('All libraries imported successfully!')

## 2. Load MRI Images
We load two sample brain MRI scans using `cv2.imread()`:
1. `sample_mri.png`: Brain slice containing a suspicious hyperintense lesion (tumor candidate).
2. `sample_mri_normal.png`: Normal brain slice without tumor for comparative validation.

In [ ]:
# Load images
mri_tumor = cv2.imread('sample_data/sample_mri.png')
mri_normal = cv2.imread('sample_data/sample_mri_normal.png')

print(f'Tumor MRI Shape: {mri_tumor.shape}')
print(f'Normal MRI Shape: {mri_normal.shape}')

# Display original images
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(cv2.cvtColor(mri_tumor, cv2.COLOR_BGR2RGB))
axes[0].set_title('Sample Brain MRI (Tumor Case)')
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(mri_normal, cv2.COLOR_BGR2RGB))
axes[1].set_title('Sample Brain MRI (Normal Case)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 3. MRI Preprocessing & Contrast Enhancement
The preprocessing pipeline executes:
1. **`cv2.resize()`**: Standardizes image dimensions to 256x256.
2. **`cv2.cvtColor()`**: Converts BGR to single-channel Grayscale.
3. **`cv2.GaussianBlur()`**: Suppresses high-frequency sensor acquisition noise.
4. **`cv2.createCLAHE()`**: Contrast Limited Adaptive Histogram Equalization sharpens subtle lesion boundaries without over-amplifying background noise.

In [ ]:
def preprocess_mri(image_bgr):
    resized = cv2.resize(image_bgr, (256, 256))
    if len(resized.shape) == 3:
        gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
    else:
        gray = resized
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(blurred)
    return gray, blurred, enhanced

gray_t, blurred_t, enhanced_t = preprocess_mri(mri_tumor)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(gray_t)
axes[0].set_title('1. Grayscale (256x256)')
axes[0].axis('off')

axes[1].imshow(blurred_t)
axes[1].set_title('2. Gaussian Blur (5x5 kernel)')
axes[1].axis('off')

axes[2].imshow(enhanced_t)
axes[2].set_title('3. CLAHE Contrast Enhanced')
axes[2].axis('off')
plt.tight_layout()
plt.show()

## 4. MRI Segmentation (Thresholding)
In T1-contrast or T2/FLAIR brain MRI, tumor tissue exhibits **hyperintensity** (abnormally bright pixel values).
We use `cv2.threshold()` to separate suspicious high-intensity regions from normal parenchyma.

In [ ]:
def segment_mri(enhanced_img, method="otsu", threshold_val=160):
    if method == "otsu":
        tissue_pixels = enhanced_img[enhanced_img > 40]
        if len(tissue_pixels) > 0:
            calc_thresh, _ = cv2.threshold(tissue_pixels, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            calc_thresh = max(int(calc_thresh), 155)
        else:
            calc_thresh = 160
        _, mask = cv2.threshold(enhanced_img, calc_thresh, 255, cv2.THRESH_BINARY)
    else:
        calc_thresh = threshold_val
        _, mask = cv2.threshold(enhanced_img, threshold_val, 255, cv2.THRESH_BINARY)
    return mask, calc_thresh

mask_t, used_thresh = segment_mri(enhanced_t, method="otsu")
print(f'Computed Segmentation Threshold: {used_thresh}')

plt.figure(figsize=(5, 5))
plt.imshow(mask_t)
plt.title(f'Binary Segmentation Mask (Threshold = {used_thresh})')
plt.axis('off')
plt.show()

## 5. Morphological Processing
Binary thresholding can produce isolated noise pixels or internal holes. We apply:
- **Opening (`cv2.MORPH_OPEN`)**: Erosion followed by dilation removes small noise.
- **Closing (`cv2.MORPH_CLOSE`)**: Dilation followed by erosion seals internal pinholes.

In [ ]:
def apply_morphology(mask, kernel_size=(5, 5)):
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, kernel_size)
    opened = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel)
    return closed

morph_t = apply_morphology(mask_t)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(mask_t)
axes[0].set_title('Raw Threshold Mask')
axes[0].axis('off')

axes[1].imshow(morph_t)
axes[1].set_title('After Morphological Opening & Closing')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 6. Contour Detection & Tumor Candidate Localization
We detect continuous region boundaries with `cv2.findContours()`, calculate the candidate area with `cv2.contourArea()`, and extract the minimal bounding box with `cv2.boundingRect()`.

In [ ]:
def detect_candidate(morph_mask, original_bgr, min_area=300):
    contours, _ = cv2.findContours(morph_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    img_h, img_w = morph_mask.shape
    total_area = float(img_h * img_w)
    
    valid_candidates = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if min_area <= area <= (0.40 * total_area):
            valid_candidates.append((cnt, area))
            
    overlay = original_bgr.copy()
    if not valid_candidates:
        return None, overlay, 0.0, 0.0, None
        
    valid_candidates.sort(key=lambda x: x[1], reverse=True)
    best_cnt, best_area = valid_candidates[0]
    x, y, w, h = cv2.boundingRect(best_cnt)
    area_pct = (best_area / total_area) * 100.0
    
    # Draw contour in Red and bounding box in Green
    cv2.drawContours(overlay, [best_cnt], -1, (0, 0, 255), 2)
    cv2.rectangle(overlay, (x, y), (x + w, y + h), (0, 255, 0), 2)
    return best_cnt, overlay, best_area, area_pct, (x, y, w, h)

cnt_t, overlay_t, area_t, area_pct_t, bbox_t = detect_candidate(morph_t, cv2.resize(mri_tumor, (256, 256)))
print(f'Detected Candidate Area: {area_t:.1f} pixels ({area_pct_t:.2f}% of image)')
print(f'Bounding Box (x, y, w, h): {bbox_t}')

plt.figure(figsize=(6, 6))
plt.imshow(cv2.cvtColor(overlay_t, cv2.COLOR_BGR2RGB))
plt.title('Tumor Candidate Localization (Red Outline + Green BBox)')
plt.axis('off')
plt.show()

## 7. Feature Extraction
We extract explainable features to numerically characterize the detected candidate:
- **Mean Intensity** & **Standard Deviation**
- **Area** & **Area Ratio (Solidity)**
- **Bounding Box Width, Height, and Aspect Ratio**

In [ ]:
def extract_features(gray_img, contour, bbox):
    if contour is None or bbox is None:
        return {
            "mean_intensity": 0.0, "std_intensity": 0.0, "area": 0.0,
            "contour_area_ratio": 0.0, "bbox_width": 0.0, "bbox_height": 0.0,
            "aspect_ratio": 0.0, "has_candidate": False
        }
    x, y, w, h = bbox
    area = float(cv2.contourArea(contour))
    bbox_area = float(w * h)
    solidity = (area / bbox_area) if bbox_area > 0 else 0.0
    aspect_ratio = (float(w) / float(h)) if h > 0 else 0.0
    
    mask = np.zeros(gray_img.shape, dtype=np.uint8)
    cv2.drawContours(mask, [contour], -1, 255, -1)
    mean_val, std_val = cv2.meanStdDev(gray_img, mask=mask)
    
    return {
        "mean_intensity": round(float(mean_val[0][0]), 2),
        "std_intensity": round(float(std_val[0][0]), 2),
        "area": round(area, 2),
        "contour_area_ratio": round(solidity, 4),
        "bbox_width": float(w),
        "bbox_height": float(h),
        "aspect_ratio": round(aspect_ratio, 4),
        "has_candidate": True
    }

features_tumor = extract_features(gray_t, cnt_t, bbox_t)
df_features = pd.DataFrame([features_tumor])
print('Extracted Candidate Features:')
df_features

## 8. Optional MRI ML Classification
We train an explainable **Decision Tree Classifier** (`DecisionTreeClassifier`) using the extracted features.
Decision Trees are chosen because their decision paths can be rendered as transparent if-else rules that can be verified by faculty examiners.

In [ ]:
# Load pre-trained model or train on synthetic feature distributions
with open('models/model.pkl', 'rb') as f:
    model_data = pickle.load(f)

clf = model_data['model']
feature_names = model_data['feature_names']
classes = model_data['classes']

# Predict on sample tumor image
feature_vector = [[features_tumor[col] for col in feature_names]]
pred_label_idx = clf.predict(feature_vector)[0]
print(f'Prediction on Tumor MRI: {classes[pred_label_idx]}')

# Display Explainable Decision Tree Rules
print('\nDecision Tree Rules:')
print(export_text(clf, feature_names=feature_names))

## 9. EEG Data Loading
We load a multi-channel EEG CSV recording (`sample_eeg.csv`).
The data includes standard 10-20 channels (`Fp1`, `Fp2`, `C3`, `C4`, `Cz`, `Pz`) sampled at 256 Hz.

In [ ]:
eeg_df = pd.read_csv('sample_data/sample_eeg.csv')
print(f'EEG Data Shape: {eeg_df.shape}')
print('Columns:', eeg_df.columns.tolist())
eeg_df.head()

## 10. EEG Signal Visualization
We visualize the raw time-domain EEG waveforms for frontal (`Fp1`), central (`C3`), and parietal (`Pz`) channels.

In [ ]:
t = eeg_df['Time_sec'].values

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
axes[0].plot(t, eeg_df['Fp1'].values, color='#1E40AF', lw=1.1)
axes[0].set_ylabel('Fp1 (uV)')
axes[0].set_title('Frontal Channel Fp1 (Beta/Theta Activity)')
axes[0].grid(True, alpha=0.4)

axes[1].plot(t, eeg_df['C3'].values, color='#047857', lw=1.1)
axes[1].set_ylabel('C3 (uV)')
axes[1].set_title('Central Channel C3')
axes[1].grid(True, alpha=0.4)

axes[2].plot(t, eeg_df['Pz'].values, color='#B91C1C', lw=1.1)
axes[2].set_ylabel('Pz (uV)')
axes[2].set_xlabel('Time (seconds)')
axes[2].set_title('Parietal Channel Pz (Dominant Alpha Rhythm)')
axes[2].grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

## 11. EEG Feature Extraction & Spectral Analysis
We compute time-domain metrics (Mean, Std Dev, Variance, Min, Max, RMS) and frequency-domain properties via NumPy FFT.

In [ ]:
def extract_eeg_features(signal, fs=256):
    sig = np.array(signal, dtype=np.float64)
    sig_mean = float(np.mean(sig))
    sig_std = float(np.std(sig))
    sig_var = float(np.var(sig))
    sig_min = float(np.min(sig))
    sig_max = float(np.max(sig))
    sig_rms = float(np.sqrt(np.mean(sig ** 2)))
    
    # FFT Spectrum
    n = len(sig)
    freqs = np.fft.rfftfreq(n, d=1.0 / fs)
    fft_vals = np.abs(np.fft.rfft(sig))
    valid_idx = np.where((freqs >= 0.5) & (freqs <= 35.0))[0]
    dom_freq = float(freqs[valid_idx[np.argmax(fft_vals[valid_idx])]])
    
    return {
        'mean': round(sig_mean, 4),
        'std': round(sig_std, 4),
        'variance': round(sig_var, 4),
        'min': round(sig_min, 4),
        'max': round(sig_max, 4),
        'rms': round(sig_rms, 4),
        'dominant_freq_hz': round(dom_freq, 2)
    }

pz_stats = extract_eeg_features(eeg_df['Pz'].values)
print('Extracted EEG Features for Channel Pz:')
pd.DataFrame([pz_stats])

## 12. Optional EEG Machine Learning
We demonstrate a lightweight Decision Tree classifier on EEG channel feature vectors to identify physiological rhythm states (e.g., Alpha dominance vs Beta activation).

In [ ]:
# Extract features for all EEG channels
channel_features = []
channels = ['Fp1', 'Fp2', 'C3', 'C4', 'Cz', 'Pz']
for ch in channels:
    feat = extract_eeg_features(eeg_df[ch].values)
    feat['channel'] = ch
    channel_features.append(feat)

df_eeg_summary = pd.DataFrame(channel_features)
print('EEG Multi-Channel Comparison:')
df_eeg_summary

## 13. Final Results & Comparative Discussion
We execute comparative analysis on both the **Tumor Case** and the **Normal Case** to confirm system reliability.

In [ ]:
# Process Normal MRI
gray_n, blurred_n, enhanced_n = preprocess_mri(mri_normal)
mask_n, _ = segment_mri(enhanced_n, method="otsu")
morph_n = apply_morphology(mask_n)
cnt_n, overlay_n, area_n, area_pct_n, bbox_n = detect_candidate(morph_n, cv2.resize(mri_normal, (256, 256)))
features_normal = extract_features(gray_n, cnt_n, bbox_n)

# ML Predictions
feat_vec_n = [[features_normal[col] for col in feature_names]]
pred_n = classes[clf.predict(feat_vec_n)[0]]

# Comparison Table
comparison_df = pd.DataFrame([
    {
        'Scan': 'Tumor MRI',
        'Candidate Detected': features_tumor['has_candidate'],
        'Candidate Area (px)': area_t,
        'Mean Intensity': features_tumor['mean_intensity'],
        'ML Prediction': classes[pred_label_idx]
    },
    {
        'Scan': 'Normal MRI',
        'Candidate Detected': features_normal['has_candidate'],
        'Candidate Area (px)': area_n,
        'Mean Intensity': features_normal['mean_intensity'],
        'ML Prediction': pred_n
    }
])

print('=== FINAL CIE-2 DEMONSTRATION SUMMARY ===')
print(comparison_df.to_string(index=False))

# Side by side final results
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(cv2.cvtColor(overlay_t, cv2.COLOR_BGR2RGB))
axes[0].set_title(f'Tumor MRI -> {classes[pred_label_idx]}')
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(overlay_n, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'Normal MRI -> {pred_n}')
axes[1].axis('off')
plt.tight_layout()
plt.show()